In [2]:
!pip install gensim

   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   - -------------------------------------- 1.0/24.4 MB 7.9 MB/s eta 0:00:03
   ---- ----------------------------------- 2.9/24.4 MB 8.6 MB/s eta 0:00:03
   ------- -------------------------------- 4.5/24.4 MB 8.9 MB/s eta 0:00:03
   ---------- ----------------------------- 6.6/24.4 MB 8.6 MB/s eta 0:00:03
   ------------ --------------------------- 7.3/24.4 MB 8.7 MB/s eta 0:00:02
   -------------- ------------------------- 8.7/24.4 MB 7.2 MB/s eta 0:00:03
   ----------------- ---------------------- 10.5/24.4 MB 7.5 MB/s eta 0:00:02
   -------------------- ------------------- 12.3/24.4 MB 7.6 MB/s eta 0:00:02
   ---------------------- ----------------- 13.6/24.4 MB 7.7 MB/s eta 0:00:02
   ---------------------- ----------------- 13.6/24.4 MB 7.7 MB/s eta 0:00:02
   ------------------------ --------------- 14.7/24.4 MB 6.5 MB/s eta 0:00:02
   --------------------------- ------------ 16.5/24.4 MB 6.7 MB/s eta 0:00:02


In [4]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# ==========================================
# CÓDIGO BASE (Datos ampliados para estratificación)
# ==========================================
def cargar_datos():
    # Simulación de datos de reseñas ampliada para evitar errores de clases con 1 solo miembro
    datos = {
        'reseña': [
            "Excelente producto, cumple con todas las expectativas",
            "Mala calidad, no lo recomiendo",
            "Regular, podría ser mejor",
            "Increíble, superó mis expectativas",
            "Pésimo servicio al cliente",
            "No es nada malo, me gustó bastante :)",
            "No volvería a comprar esto, horrible",
            "Me encanta, funciona de maravilla :D",
            "Un producto bastante promedio, ni fu ni fa",
            "No está mal, cumple su función básica sin destacar",
            "Horrible, dejó de funcionar al segundo día :(",
            "Bastante aceptable por el precio que tiene"
        ],
        'sentimiento': [
            'positivo', 'negativo', 'neutral', 'positivo', 'negativo', 
            'positivo', 'negativo', 'positivo', 'neutral', 'neutral', 
            'negativo', 'neutral'
        ]
    }
    return pd.DataFrame(datos)

df = cargar_datos()
print("Datos cargados exitosamente:\n")
print(df.head())
print("-" * 50)



# TAREA 1: Preprocesamiento de Texto

def expandir_negaciones(texto):
    """
    Expande negaciones básicas en español uniendo 'no' con la palabra siguiente 
    para conservar el contexto semántico (ej. 'no recomiendo' -> 'no_recomiendo').
    """
    texto = re.sub(r'\bno\s+(\w+)', r'no_\1', texto, flags=re.IGNORECASE)
    return texto

def limpiar_texto(texto):
    """
    Pipeline de limpieza y normalización:
    - Minúsculas
    - Manejo básico de emoticonos comunes
    - Expansión de negaciones
    - Eliminación de caracteres especiales y números
    """
    # 1. Convertir a minúsculas
    texto = texto.lower()
    
    # 2. Manejo de emoticonos comunes
    texto = texto.replace(":)", " positivo_emoji ")
    texto = texto.replace(":D", " positivo_emoji ")
    texto = texto.replace(":(", " negativo_emoji ")
    
    # 3. Aplicar expansión de negaciones
    texto = expandir_negaciones(texto)
    
    # 4. Remover caracteres especiales, puntuación y números (manteniendo guiones bajos)
    texto = re.sub(r'[^a-záéíóúñ_\s]', '', texto)
    
    # 5. Normalizar espacios en blanco
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    return texto

# Aplicar preprocesamiento al DataFrame
df['reseña_limpia'] = df['reseña'].apply(limpiar_texto)
print("Ejemplo de texto preprocesado:\n")
print(df[['reseña', 'reseña_limpia']].head())
print("-" * 50)


# TAREA 2: Extracción de Características (TF-IDF)

vectorizador_tfidf = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
X_tfidf = vectorizador_tfidf.fit_transform(df['reseña_limpia'])
y = df['sentimiento']

print(f"Dimensiones de la matriz de características TF-IDF: {X_tfidf.shape}")
print("-" * 50)



# TAREA 3: Modelado, Evaluación y Validación Cruzada


# 1. División de datos 
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.3, random_state=42, stratify=y
)

# 2. Entrenamiento de Múltiples Modelos
modelos = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Regresión Logística": LogisticRegression(random_state=42)
}

for nombre, modelo in modelos.items():
    print(f"\n--- Evaluando Modelo: {nombre} ---")
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    
    print("Reporte de Clasificación:")
    print(classification_report(y_test, y_pred, zero_division=0))

# 3. Implementación de Validación Cruzada (Cross-Validation)
print("\n--- Validación Cruzada (Stratified K-Fold) ---")
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
modelo_cv = LogisticRegression(random_state=42)

scores = cross_val_score(modelo_cv, X_tfidf, y, cv=cv, scoring='accuracy')
print(f"Scores de Accuracy por fold: {scores}")
print(f"Accuracy Promedio en Validación Cruzada: {scores.mean():.2f} (+/- {scores.std():.2f})")

Datos cargados exitosamente:

                                              reseña sentimiento
0  Excelente producto, cumple con todas las expec...    positivo
1                     Mala calidad, no lo recomiendo    negativo
2                          Regular, podría ser mejor     neutral
3                 Increíble, superó mis expectativas    positivo
4                         Pésimo servicio al cliente    negativo
--------------------------------------------------
Ejemplo de texto preprocesado:

                                              reseña  \
0  Excelente producto, cumple con todas las expec...   
1                     Mala calidad, no lo recomiendo   
2                          Regular, podría ser mejor   
3                 Increíble, superó mis expectativas   
4                         Pésimo servicio al cliente   

                                       reseña_limpia  
0  excelente producto cumple con todas las expect...  
1                      mala calidad no_lo recomien

In [ ]:
#Actividad 17 
# TAREA 1: Diseño de Intenciones (Patrones y Respuestas)
INTENCIONES = {
    "saludo": {
        "patrones": ["hola", "buenos días", "buenas tardes"],
        "respuesta": "¡Hola! ¿En qué puedo ayudarte hoy?"
    },
    "horario": {
        "patrones": ["horario", "que hora abren", "atienden"],
        "respuesta": "Nuestro horario es de lunes a viernes de 9:00 a 18:00 hrs."
    },
    "contacto": {
        "patrones": ["contacto", "telefono", "correo", "humano"],
        "respuesta": "Puedes contactarnos en soporte@empresa.com o al 555-1234."
    }
}

# TAREA 2: Procesamiento de Consultas y Fallback
def procesar_consulta(mensaje):
    mensaje = mensaje.lower().strip()
    
    # Sistema de coincidencia de patrones
    for intencion, datos in INTENCIONES.items():
        for patron in datos["patrones"]:
            if patron in mensaje:
                return datos["respuesta"]
                
    # Sistema de Fallback para consultas no entendidas
    return "Lo siento, no entendí tu pregunta. Intenta consultar sobre 'horario' o 'contacto'."

# TAREA 3: Interfaz de Usuario, Historial y Evaluación
def iniciar_chatbot():
    print("🤖 Chatbot: ¡Hola! Escribe tu consulta (o escribe 'salir' para terminar).\n")
    
    historial = [] # Almacena el historial de la conversación
    
    while True:
        usuario = input("Tú: ")
        
        if usuario.lower() == 'salir':
            break
            
        # Guardar en historial
        historial.append(f"Usuario: {usuario}")
        
        # Procesar y responder
        respuesta = procesar_consulta(usuario)
        print(f"Chatbot: {respuesta}\n")
        
        # Guardar respuesta en historial
        historial.append(f"Chatbot: {respuesta}")
        
    # Evaluación de satisfacción al finalizar
    print("\n--- Evaluación de Satisfacción ---")
    calificacion = input("¿Cómo calificarías la atención de 1 a 5 estrellas?: ")
    print(f"¡Gracias por tu respuesta! (Calificación registrada: {calificacion} estrellas).")
    print(f"Total de mensajes en el historial: {len(historial)}")

# Ejecutar el programa
if __name__ == "__main__":
    iniciar_chatbot()

🤖 Chatbot: ¡Hola! Escribe tu consulta (o escribe 'salir' para terminar).



Tú:  horario 


Chatbot: Nuestro horario es de lunes a viernes de 9:00 a 18:00 hrs.



Tú:  contacto 


Chatbot: Puedes contactarnos en soporte@empresa.com o al 555-1234.



Tú:  dirección 


Chatbot: Lo siento, no entendí tu pregunta. Intenta consultar sobre 'horario' o 'contacto'.



Tú:  salir 


Chatbot: Lo siento, no entendí tu pregunta. Intenta consultar sobre 'horario' o 'contacto'.

